# Hurricane Laura (2020): MOM6 vs in situ observations with DART model-to-observation interpolation

Observations near Laura's track (20-28 Aug 2020) are written to DART `obs_seq` files, and
[model2obs](https://github.com/CROCODILE-CESM/model2obs) runs DART's `perfect_model_obs` to interpolate
each MOM6 experiment (JRA55 vs ERA5 DATM forcing) to every observation location and time.

- **Model**: the `.003` runs, 3-hourly mean `thetao`, `so`, `uo`, `vo` and layer thickness `h` on the native
  MOM6 layers (`mom6.h.native3h`). DART interpolates horizontally on the MOM6 grid and vertically using the
  layer thicknesses (observations below the model bathymetry fail QC); potential temperature is converted to
  in situ temperature. The same fields on a fine z grid are in `mom6.h.zfine` for plotting.
- **Observations** (same selection as the diagnostics notebook, within `TRACK_DISTANCE_DEG` of Laura):
  IOOS gliders (2 m bins per profile), Argo profiles (adjusted values, QC 1-2), NDBC buoy water temperature
  (~1 m) and the Stone Mooring ADCP currents (NDBC 42395).
- **Matching**: one `obs_seq` file per 3-hour window centred on each model output time; model2obs pairs
  each model time with the file inside its `TIME_WINDOW`.

Run this notebook with the `model2obs` kernel (DART paths are set when the environment is activated).

In [ ]:
import os
import gzip
import re
from pathlib import Path

import cartopy.crs as ccrs
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import yaml

from model2obs.workflows import WorkflowModelObs

WORKSPACE = Path("/glade/u/home/shanru/work/crocodile2026/workspace")
DATA_DIR = WORKSPACE / "data"
ARCHIVE = Path(os.environ["SCRATCH"]) / "archive"
EXPS = {
    "JRA55": "hurcn_gomcar_ocn_wav.003",
    "ERA5":  "hurcn_gomcar_ocn_wav_era5.003",
}
MODEL_STREAM = "mom6.h.native3h"
M2O_DIR = WORKSPACE / "model2obs_laura"
PERFECT_MODEL_OBS_DIR = Path(os.environ["DART_ROOT_PATH"]) / "models" / "MOM6" / "work"

T0, T1 = "2020-08-20", "2020-08-28"
TRACK_DISTANCE_DEG = 2.0
OBS_BIN = pd.Timedelta(hours=3)   # model output interval; one obs_seq file per output time
SOURCE_COLOR = {"JRA55": "tab:blue", "ERA5": "tab:red"}
plt.rcParams["font.size"] = 12

# Laura best track (IBTrACS v04r01, downloaded by the diagnostics notebook)
ib = pd.read_csv(DATA_DIR / "ibtracs" / "ibtracs.LAURA_2020.v04r01.csv", skiprows=[1], low_memory=False)
TRACK = pd.DataFrame({"time": pd.to_datetime(ib["ISO_TIME"]),
                      "lat": ib["LAT"].astype(float), "lon": ib["LON"].astype(float)})


def distance_to_track_deg(lon, lat):
    """Distance (degrees of latitude, longitude scaled by cos(lat)) to the nearest IBTrACS fix of Laura."""
    lon, lat = np.atleast_1d(np.asarray(lon, float)), np.atleast_1d(np.asarray(lat, float))
    dlon = (lon[:, None] - TRACK["lon"].values[None, :]) * np.cos(np.deg2rad(lat))[:, None]
    dlat = lat[:, None] - TRACK["lat"].values[None, :]
    return np.hypot(dlon, dlat).min(axis=1)


def closest_approach(lon, lat):
    """Time of Laura's nearest IBTrACS fix to a point."""
    d = np.hypot((TRACK["lon"] - lon) * np.cos(np.deg2rad(lat)), TRACK["lat"] - lat)
    return TRACK["time"].iloc[int(np.argmin(d))]


def pressure_to_depth(p, lat):
    """Depth (m) from pressure (dbar), Saunders (1981)."""
    c1 = (5.92 + 5.25 * np.sin(np.deg2rad(lat)) ** 2) * 1e-3
    return (1 - c1) * p - 2.21e-6 * p**2

## Observations

One row per observation: DART observation type, time, position, depth (m, positive down) and value
(°C, psu, m/s).

In [ ]:
NA = [99.0, 999.0, 9999.0]
obs = []

# Gliders: profiles from the IOOS Glider DAC cache, gliders coming within TRACK_DISTANCE_DEG of Laura,
# averaged in 2 m depth bins
gl = pd.read_csv(DATA_DIR / "gliders" / f"ioos_gliders_{T0}_{T1}.csv", parse_dates=["time"])
first = gl.groupby(["source", "profile_id"])[["lon", "lat"]].first()
near = (pd.Series(distance_to_track_deg(first["lon"], first["lat"]), index=first.index)
        .groupby(level="source").min() <= TRACK_DISTANCE_DEG)
gl = gl[gl["source"].isin(near[near].index)].dropna(subset=["depth"])
gl["zbin"] = (gl["depth"] // 2.0) * 2.0 + 1.0
prof = gl.groupby(["source", "profile_id"]).agg(time=("time", "median"), lon=("lon", "mean"), lat=("lat", "mean"))
binned = gl.groupby(["source", "profile_id", "zbin"])[["temp", "salt"]].mean().reset_index()
binned = binned.join(prof, on=["source", "profile_id"])
for var, otype in (("temp", "GLIDER_TEMPERATURE"), ("salt", "GLIDER_SALINITY")):
    b = binned.dropna(subset=[var])
    obs.append(pd.DataFrame({"type": otype, "platform": "Glider", "source": b["source"].str.split("-").str[0],
                             "time": b["time"], "lon": b["lon"], "lat": b["lat"], "depth": b["zbin"],
                             "value": b[var]}))

# Argo: profiles in the GDAC index, adjusted values where available, QC flags 1-2
index = pd.read_csv(DATA_DIR / "argo" / "ar_index_global_prof.txt.gz", comment="#", dtype={"date": str})
index["time"] = pd.to_datetime(index["date"], format="%Y%m%d%H%M%S", errors="coerce")
index = index[index["time"].between(pd.Timestamp(T0), pd.Timestamp(f"{T1} 23:59:59"))].dropna(subset=["latitude"])
index = index[distance_to_track_deg(index["longitude"], index["latitude"]) <= TRACK_DISTANCE_DEG]
for _, row in index.iterrows():
    with xr.open_dataset(DATA_DIR / "argo" / "profiles" / Path(row["file"]).name, decode_times=False) as ds:
        vals = {}
        for var in ("PRES", "TEMP", "PSAL"):
            adj = f"{var}_ADJUSTED"
            name = adj if adj in ds and np.isfinite(ds[adj].values[0]).any() else var
            v = ds[name].values[0].astype(float)
            v[~np.isin(ds[f"{name}_QC"].values[0].astype(str), ["1", "2"])] = np.nan
            vals[var] = v
    depth = pressure_to_depth(vals["PRES"], row["latitude"])
    for var, otype in (("TEMP", "ARGO_TEMPERATURE"), ("PSAL", "ARGO_SALINITY")):
        ok = np.isfinite(depth) & np.isfinite(vals[var])
        obs.append(pd.DataFrame({"type": otype, "platform": "Argo", "source": row["file"].split("/")[1],
                                 "time": row["time"], "lon": row["longitude"], "lat": row["latitude"],
                                 "depth": depth[ok], "value": vals[var][ok]}))


def read_ndbc(path):
    """NDBC standard meteorological or ADCP text file (gzip) with a datetime index."""
    df = pd.read_csv(path, sep=r"\s+", skiprows=2, header=None, na_values=NA)
    names = pd.read_csv(path, sep=r"\s+", nrows=0).columns.tolist()
    # ADCP headers can list fewer depth bins than each record carries
    if df.shape[1] > len(names) and names[5] == "DEP01":
        nbin = (df.shape[1] - 5) // 3
        names = names[:5] + [f"{v}{k:02d}" for k in range(1, nbin + 1) for v in ("DEP", "DIR", "SPD")]
    df.columns = names
    df.index = pd.to_datetime(df.iloc[:, :5].set_axis(["year", "month", "day", "hour", "minute"], axis=1))
    return df.iloc[:, 5:]


# NDBC buoys: hourly water temperature (~1 m) at buoys within TRACK_DISTANCE_DEG of Laura
stations = []
for line in open(DATA_DIR / "ndbc" / "station_table.txt"):
    f = line.split("|")
    m = re.match(r"\s*([\d.]+)\s*([NS])\s*([\d.]+)\s*([EW])", f[6]) if not line.startswith("#") else None
    if m and "buoy" in f[2].lower():
        stations.append((f[0].lower(), float(m[1]) * (1 if m[2] == "N" else -1),
                         float(m[3]) * (1 if m[4] == "E" else -1)))
for station, lat, lon in stations:
    path = DATA_DIR / "ndbc" / "stdmet" / f"{station}h2020.txt.gz"
    if not path.exists() or distance_to_track_deg(lon, lat)[0] > TRACK_DISTANCE_DEG:
        continue
    wtmp = read_ndbc(path)["WTMP"].loc[T0:f"{T1} 23:59"].dropna()
    obs.append(pd.DataFrame({"type": "MOORING_TEMPERATURE", "platform": "Buoy", "source": station,
                             "time": wtmp.index, "lon": lon, "lat": lat, "depth": 1.0, "value": wtmp.values}))

# Stone Mooring (NDBC 42395, 26.4N 90.8W): hourly ADCP currents
STONE = dict(lat=26.4, lon=-90.8)
adcp = read_ndbc(DATA_DIR / "stone_mooring" / "42395a2020.txt.gz").loc[T0:f"{T1} 23:59"]
spd = adcp.filter(like="SPD").values / 100.0
rad = np.deg2rad(adcp.filter(like="DIR").values)
dep = adcp.filter(like="DEP").values
for comp, val in (("U", spd * np.sin(rad)), ("V", spd * np.cos(rad))):
    t = np.repeat(adcp.index.values, dep.shape[1])
    ok = np.isfinite(val.ravel()) & np.isfinite(dep.ravel())
    obs.append(pd.DataFrame({"type": f"ADCP_{comp}_CURRENT_COMPONENT", "platform": "Stone Mooring",
                             "source": "42395", "time": t[ok], "lon": STONE["lon"], "lat": STONE["lat"],
                             "depth": dep.ravel()[ok], "value": val.ravel()[ok]}))

OBS = pd.concat(obs, ignore_index=True)
OBS = OBS[OBS["time"].between(pd.Timestamp(T0), pd.Timestamp(f"{T1} 23:59:59"))].reset_index(drop=True)
print(OBS.groupby(["platform", "type"]).agg(n=("value", "size"), sources=("source", "nunique")).to_string())

## DART obs_seq files

ASCII `obs_seq` format: location in radians with depth in metres (`which_vert = 3`), time as days and
seconds since 1601-01-01, salinity in kg/kg (psu/1000) as DART expects. Observation error variances only
scale model2obs' normalized residuals.

In [ ]:
OBS_ERROR_SD = {"TEMPERATURE": 0.5, "SALINITY": 0.1e-3, "CURRENT_COMPONENT": 0.1}
DART_EPOCH = np.datetime64("1601-01-01T00:00:00", "s")
OBS_SEQ_DIR = M2O_DIR / "obs_seq_in"


def write_obs_seq(df, path):
    """Write observations (type, time, lon, lat, depth, value) as a DART ASCII obs_seq file."""
    types = sorted(df["type"].unique())
    kind = {t: k + 1 for k, t in enumerate(types)}
    value = np.where(df["type"].str.endswith("SALINITY"), df["value"] / 1000.0, df["value"])
    err = df["type"].map(lambda t: next(sd for key, sd in OBS_ERROR_SD.items() if t.endswith(key))) ** 2
    since = (df["time"].values.astype("datetime64[s]") - DART_EPOCH).astype(np.int64)
    days, secs = since // 86400, since % 86400
    lon = np.deg2rad(np.mod(df["lon"].values, 360.0))
    lat = np.deg2rad(df["lat"].values)
    n = len(df)
    lines = [" obs_sequence", "obs_type_definitions", f"{len(types):11d}"]
    lines += [f"{kind[t]:11d} {t}" for t in types]
    lines += [f"num_copies:{1:12d} num_qc:{1:12d}", f"num_obs:{n:12d} max_num_obs:{n:12d}",
              "observation", "QC", f"first:{1:12d} last:{n:12d}"]
    for i, (otype, d) in enumerate(zip(df["type"].values, df["depth"].values)):
        k = i + 1
        lines += [f"OBS {k:12d}", repr(float(value[i])), "1",
                  f"{k - 1 if k > 1 else -1:12d}{k + 1 if k < n else -1:12d}{-1:12d}",
                  "obdef", "loc3d", f"{float(lon[i])!r} {float(lat[i])!r} {float(d)!r} 3", "kind",
                  f"{kind[otype]:12d}",
                  f"{secs[i]} {days[i]}", repr(float(err.values[i]))]
    Path(path).write_text("\n".join(lines) + "\n")


def write_obs_seq_files(obs_df, outdir, bin_width=OBS_BIN):
    """One obs_seq file per bin_width window (aligned to 00 UTC), in time order within each file."""
    outdir.mkdir(parents=True, exist_ok=True)
    for f in outdir.glob("obs_seq.*.in"):
        f.unlink()
    bins = obs_df["time"].dt.floor(bin_width)
    for start, g in obs_df.groupby(bins):
        write_obs_seq(g.sort_values("time"), outdir / f"obs_seq.{start:%Y%m%d%H}.in")
    return sorted(outdir.glob("obs_seq.*.in"))


obs_seq_files = write_obs_seq_files(OBS, OBS_SEQ_DIR)
print(f"{len(obs_seq_files)} obs_seq files in {OBS_SEQ_DIR}, {len(OBS)} observations")

## Interpolate each experiment with model2obs (DART perfect_model_obs)

The model files are the `mom6.h.native3h` daily files (eight 3-hourly means each). model2obs pairs each
model time with the `obs_seq` file falling inside `TIME_WINDOW` around it (all-or-nothing), runs
`perfect_model_obs`, and writes the model-observation pairs to Parquet.

In [ ]:
TIME_WINDOW = dict(hours=3)
USE_THESE_OBS = sorted(OBS["type"].unique())
MODEL_STATE_VARIABLES = {"thetao": "QTY_POTENTIAL_TEMPERATURE", "so": "QTY_SALINITY",
                         "uo": "QTY_U_CURRENT_COMPONENT", "vo": "QTY_V_CURRENT_COMPONENT",
                         "h": "QTY_LAYER_THICKNESS"}


def model2obs_config(name, case, model_glob=f"*.{MODEL_STREAM}.*.nc"):
    """Write the model2obs YAML configuration for one experiment and return its path."""
    hist = ARCHIVE / case / "ocn" / "hist"
    run_dir = M2O_DIR / name
    model_dir = run_dir / "model_files"
    model_dir.mkdir(parents=True, exist_ok=True)
    for f in model_dir.glob("*.nc"):
        f.unlink()
    model_files = sorted(hist.glob(model_glob))
    for f in model_files:
        (model_dir / f.name).symlink_to(f)
    out = {k: run_dir / k for k in ("out_obs_seq", "input_nml_bck", "trimmed_obs", "parquet", "tmp")}
    for p in out.values():
        if p.exists():
            for f in sorted(p.rglob("*"), reverse=True):
                f.unlink() if f.is_file() or f.is_symlink() else f.rmdir()
    config = {
        "ocean_model": "MOM6",
        "model_files_folder": str(model_dir),
        "template_file": str(model_files[0]),
        "static_file": str(hist / f"{case}.mom6.h.static.nc"),
        "ocean_geometry": str(hist / f"{case}.mom6.h.ocean_geometry.nc"),
        "obs_seq_in_folder": str(OBS_SEQ_DIR),
        "output_folder": str(out["out_obs_seq"]),
        "input_nml_bck": str(out["input_nml_bck"]),
        "trimmed_obs_folder": str(out["trimmed_obs"]),
        "parquet_folder": str(out["parquet"]),
        "tmp_folder": str(out["tmp"]),
        "log_file": str(run_dir / "logs" / "model2obs.log"),
        "perfect_model_obs_dir": str(PERFECT_MODEL_OBS_DIR),
        "use_these_obs": USE_THESE_OBS,
        "time_window": TIME_WINDOW,
        "model_state_variables": MODEL_STATE_VARIABLES,
        "use_pseudo_depth": False,
        "layer_name": "zl",
    }
    path = run_dir / "config.yaml"
    path.write_text(yaml.safe_dump(config, sort_keys=False))
    return path


workflows = {}
for name, case in EXPS.items():
    workflows[name] = WorkflowModelObs.from_config_file(str(model2obs_config(name, case)))
    n = workflows[name].run(trim_obs=True)
    print(f"{name}: {n} model-obs file pairs interpolated")

## Results

model2obs reports `residual = obs - model`; DART QC <= 2 marks successful interpolation (observations
below the model bathymetry or outside the domain fail).

In [ ]:
def load_pairs(name):
    df = workflows[name].get_good_model_obs_df(compute=True)
    df = df.rename(columns={"interpolated_model": "model", "vertical": "depth"})
    df["forcing"] = name
    sal = df["type"].str.endswith("SALINITY")
    df.loc[sal, ["obs", "model"]] *= 1000.0
    df["diff"] = df["model"] - df["obs"]
    df["lon"] = ((df["longitude"] + 180.0) % 360.0) - 180.0
    return df


PAIRS = pd.concat([load_pairs(name) for name in EXPS], ignore_index=True)
QUANTITY = {"TEMPERATURE": "Temperature (\u00b0C)", "SALINITY": "Salinity (psu)",
            "CURRENT_COMPONENT": "Current (m/s)"}
PAIRS["quantity"] = PAIRS["type"].map(lambda t: next(q for q in QUANTITY if t.endswith(q)))
PAIRS["platform"] = PAIRS["type"].str.split("_").str[0].map(
    {"GLIDER": "Glider", "ARGO": "Argo", "MOORING": "Buoy", "ADCP": "Stone Mooring"})
summary = (PAIRS.groupby(["quantity", "platform", "forcing"])["diff"]
           .agg(N="size", Bias="mean", RMSE=lambda d: np.sqrt((d**2).mean()))
           .unstack("forcing").reorder_levels([1, 0], axis=1).reindex(columns=list(EXPS), level=0))
print("Model minus observation, successful DART interpolations only:")
summary.round(3)

In [ ]:
# Bias and RMSE profiles (model - obs) for temperature and salinity, upper 200 m
ZBINS = np.r_[0:50:5, 50:100:10, 100:210:20]
fig, axes = plt.subplots(1, 4, figsize=(18, 7), sharey=True, layout="constrained")
for col, (quantity, stat) in enumerate([(q, s) for q in ("TEMPERATURE", "SALINITY") for s in ("Bias", "RMSE")]):
    ax = axes[col]
    sel = PAIRS[(PAIRS["quantity"] == quantity) & (PAIRS["depth"] < ZBINS[-1])]
    for (platform, name), g in sel.groupby(["platform", "forcing"]):
        zc = pd.cut(g["depth"], ZBINS)
        d = g.groupby(zc, observed=True)["diff"]
        val = d.mean() if stat == "Bias" else np.sqrt(d.apply(lambda x: (x**2).mean()))
        mid = [iv.mid for iv in val.index]
        ax.plot(val.values, mid, color=SOURCE_COLOR[name], marker={"Glider": "^", "Argo": "o", "Buoy": "s"}[platform],
                linestyle={"Glider": "-", "Argo": "--", "Buoy": ":"}[platform], label=f"{platform} {name}")
    ax.axvline(0, color="k", linewidth=0.8)
    ax.set_xlabel(f"{stat} {QUANTITY[quantity]}")
    ax.set_title(f"({'abcd'[col]}) {QUANTITY[quantity].split(' (')[0]} {stat.lower()}", loc="left", fontweight="bold")
    ax.grid(alpha=0.3)
axes[0].invert_yaxis()
axes[0].set_ylabel("Depth (m)")
axes[0].legend(fontsize=10)
fig.suptitle("MOM6 minus observations near Laura (DART interpolation)", fontsize=15, fontweight="bold")
plt.show()

In [ ]:
# Argo floats with profiles before and after Laura passed: observed vs interpolated model temperature
argo_t = PAIRS[(PAIRS["type"] == "ARGO_TEMPERATURE")]
repeat = argo_t.groupby("source")["time"].nunique()
floats = [f for f in repeat[repeat >= 2].index][:6]
fig, axes = plt.subplots(1, len(floats), figsize=(4 * len(floats), 6.5), sharey=True, layout="constrained",
                         squeeze=False)
for ax, flt in zip(axes[0], floats):
    g = argo_t[argo_t["source"] == flt]
    for k, (t, p) in enumerate(g.groupby("time")):
        ls = "-" if k == 0 else "--"
        ref = p[p["forcing"] == "JRA55"].sort_values("depth")
        ax.plot(ref["obs"], ref["depth"], color="k", linestyle=ls, linewidth=2, label=f"Argo {t:%d %b %H}Z")
        for name in EXPS:
            m = p[p["forcing"] == name].sort_values("depth")
            ax.plot(m["model"], m["depth"], color=SOURCE_COLOR[name], linestyle=ls, linewidth=1.5,
                    label=f"{name} {t:%d %b}")
    ax.set_ylim(300, 0)
    ax.set_title(f"Float {flt}", loc="left")
    ax.set_xlabel("Temperature (\u00b0C)")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8, loc="lower right")
axes[0, 0].set_ylabel("Depth (m)")
fig.suptitle("Argo profiles before (solid) and after (dashed) vs MOM6", fontsize=15, fontweight="bold")
plt.show()

In [ ]:
# Stone Mooring ADCP: observed vs interpolated model eastward and northward currents (time-depth)
st = PAIRS[PAIRS["platform"] == "Stone Mooring"]
if not st.empty:
    t_ca = closest_approach(STONE["lon"], STONE["lat"])
    fig, axes = plt.subplots(2, 3, figsize=(20, 8), sharex=True, sharey=True, layout="constrained")
    for row, comp in enumerate(("U", "V")):
        g = st[st["type"] == f"ADCP_{comp}_CURRENT_COMPONENT"]
        panels = [("Observed", g[g["forcing"] == "JRA55"], "obs")] + [(f"MOM6 {n}", g[g["forcing"] == n], "model")
                                                                     for n in EXPS]
        for col, (title, p, field) in enumerate(panels):
            ax = axes[row, col]
            grid = p.pivot_table(index="depth", columns="time", values=field)
            pc = ax.pcolormesh(grid.columns, grid.index, grid.values, cmap="RdBu_r", vmin=-1.2, vmax=1.2,
                               shading="nearest")
            ax.axvline(t_ca, color="k", linestyle="--")
            ax.set_title(f"{title} {'eastward' if comp == 'U' else 'northward'} current", loc="left")
            ax.set_ylim(300, 0)
    for ax in axes[:, 0]:
        ax.set_ylabel("Depth (m)")
    for ax in axes[-1]:
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%d"))
        ax.set_xlabel("Day of Aug 2020")
    fig.colorbar(pc, ax=axes, label="m/s", shrink=0.8)
    fig.suptitle("Stone Mooring (NDBC 42395) currents vs MOM6 (dashed: Laura's closest approach)",
                 fontsize=15, fontweight="bold")
    plt.show()